# IRS Audit Risk Analyzer
### Federal examination rate analysis using IRS Data Book Table 17 (Tax Years 2010–2022)

**Author:** Jason Chung · UCLA Statistics & Data Science  
**Data:** IRS Data Book Publication 55B, Table 17 · IRS SOI Publication 1304  

---

## Project overview

The IRS examines a small fraction of individual returns each year — but that fraction varies dramatically by income level, filing type, and deduction patterns. This notebook:

1. Analyzes **audit rate trends** from 2010–2022 (a story of budget cuts, COVID, and IRA recovery)
2. Maps **examination rates by income bracket** from the most recent IRS Data Book
3. Builds a **weighted risk scoring model** grounded in IRS DIF score research
4. Runs **sensitivity analyses** showing how self-employment, charitable giving, and business losses shift audit probability


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "#0e0f11", "axes.facecolor": "#16181c",
    "axes.edgecolor": "#2a2d35", "axes.labelcolor": "#a0a3ad",
    "axes.titlecolor": "#f0f0ee", "xtick.color": "#7a7d86",
    "ytick.color": "#7a7d86", "text.color": "#f0f0ee",
    "grid.color": "#2a2d35", "grid.linestyle": "--", "grid.alpha": 0.6,
})
BLUE="#4d9de0"; GREEN="#3dd68c"; AMBER="#f0a500"; RED="#f25c54"; GRAY="#7a7d86"; PURPLE="#9b7ee8"

## 2. Real IRS Data — Data Book Table 17

Source: [IRS Data Book Publication 55B, Table 17](https://www.irs.gov/statistics/soi-tax-stats-irs-data-book)

Audit rates (%) for individual returns by Total Positive Income bracket, Tax Years 2010–2022.

In [ ]:
audit_rates = pd.DataFrame({
    "tax_year": [2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022],
    "overall":  [1.11,1.11,1.03,0.96,0.86,0.84,0.70,0.62,0.45,0.45,0.29,0.38,0.49],
    "under_25k":[1.92,1.85,1.62,1.44,1.21,1.14,1.00,0.97,0.69,0.69,0.51,0.54,0.61],
    "k25_50":   [0.54,0.53,0.51,0.46,0.41,0.38,0.31,0.27,0.21,0.21,0.13,0.19,0.26],
    "k50_75":   [0.43,0.43,0.41,0.37,0.33,0.30,0.25,0.21,0.17,0.17,0.10,0.15,0.21],
    "k100_200": [0.48,0.48,0.46,0.42,0.37,0.34,0.28,0.23,0.18,0.19,0.12,0.16,0.22],
    "k200_500": [0.98,1.01,0.97,0.90,0.77,0.72,0.58,0.47,0.36,0.36,0.22,0.28,0.38],
    "k500_1m":  [1.95,2.05,1.95,1.76,1.52,1.42,1.17,0.89,0.57,0.59,0.36,0.47,0.63],
    "k1m_5m":   [4.01,4.06,3.70,3.24,2.73,2.43,1.99,1.47,0.97,1.00,0.69,0.84,1.10],
    "over_10m": [18.38,18.38,16.22,14.17,10.74,9.55,6.62,4.84,3.22,3.90,2.48,3.24,4.40],
})

decline = (audit_rates["overall"].iloc[-1] - audit_rates["overall"].iloc[0]) / audit_rates["overall"].iloc[0] * 100
print(f"Overall rate 2010: {audit_rates['overall'].iloc[0]:.2f}%  |  2022: {audit_rates['overall'].iloc[-1]:.2f}%  |  Change: {decline:.1f}%")
audit_rates

## 3. Audit rate trend analysis

From 2010 to 2020, the IRS overall audit rate fell **56%** — driven by sustained budget cuts. The 2022 Inflation Reduction Act allocated $80B in new IRS funding and rates are recovering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("IRS audit rates: a decade of decline and recovery (Data Book Table 17)", y=1.01)

ax1 = axes[0]
ax1.plot(audit_rates["tax_year"], audit_rates["overall"], color=RED, linewidth=2.5, marker="o", markersize=5)
ax1.fill_between(audit_rates["tax_year"], audit_rates["overall"], alpha=0.1, color=RED)
ax1.annotate("IRA $80B\nfunding", xy=(2022,0.49), xytext=(2020,0.70),
             arrowprops=dict(arrowstyle="->", color=GREEN, lw=1.5), color=GREEN, fontsize=9)
ax1.annotate("Budget\ncuts begin", xy=(2011,1.11), xytext=(2013,1.25),
             arrowprops=dict(arrowstyle="->", color=AMBER, lw=1.5), color=AMBER, fontsize=9)
ax1.set_title("Overall audit rate 2010-2022")
ax1.set_ylabel("Audit rate (%)")
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
for col, label, color in [
    ("under_25k","<$25k",GRAY), ("k100_200","$100-200k",BLUE),
    ("k500_1m","$500k-1M",GREEN), ("k1m_5m","$1M-5M",AMBER), ("over_10m",">$10M",RED)]:
    ax2.plot(audit_rates["tax_year"], audit_rates[col], marker="o", markersize=4,
             linewidth=2, label=label, color=color)
ax2.set_title("Audit rates by income bracket")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.legend(facecolor="#1c1e24", edgecolor="#3a3d45", labelcolor="#f0f0ee")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 2022 audit rates by income — the U-shaped curve

Low-income filers face elevated rates due to EITC compliance checks. Middle-income is the safest bracket. High-income faces steep scrutiny.

In [ ]:
audit_2022 = pd.DataFrame({
    "bracket": ["<$25k","$25-50k","$50-75k","$75-100k",
                "$100-200k","$200-500k","$500k-1M","$1M-5M","$5M-10M",">$10M"],
    "rate":    [0.61, 0.26, 0.21, 0.20, 0.22, 0.38, 0.63, 1.10, 2.35, 4.40],
})

fig, ax = plt.subplots(figsize=(12, 5))
colors = [GREEN if r<0.5 else AMBER if r<1.5 else RED for r in audit_2022["rate"]]
bars = ax.bar(audit_2022["bracket"], audit_2022["rate"], color=colors, edgecolor="#0e0f11", width=0.65)
for bar, rate in zip(bars, audit_2022["rate"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04, f"{rate:.2f}%",
            ha="center", va="bottom", fontsize=9, color="#f0f0ee", fontweight="bold")
ax.set_title("Federal audit rates by income bracket — Tax Year 2022")
ax.set_ylabel("Examination coverage rate (%)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xticklabels(audit_2022["bracket"], rotation=30, ha="right")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Risk scoring model

| Factor | Weight | Source |
|---|---|---|
| Income bracket | 30% | IRS Data Book Table 17 |
| Self-employment (Sch. C) | 25% | ~2.5x examination probability |
| Charitable giving ratio | 15% | Flags when >5% of AGI |
| Business loss (Sch. C) | 15% | Repeated losses trigger scrutiny |
| Cash-intensive business | 10% | Restaurants, salons, etc. |
| Large itemized deductions | 5% | Above peer average |


In [ ]:
def audit_risk_score(agi, self_employed=False, charitable_ratio=0.0,
                     has_business_loss=False, cash_business=False, large_deductions=False):
    """Compute 0-100 audit risk score from IRS examination patterns."""
    if agi < 25000:      base = 0.61
    elif agi < 50000:    base = 0.26
    elif agi < 75000:    base = 0.21
    elif agi < 100000:   base = 0.20
    elif agi < 200000:   base = 0.22
    elif agi < 500000:   base = 0.38
    elif agi < 1000000:  base = 0.63
    elif agi < 5000000:  base = 1.10
    elif agi < 10000000: base = 2.35
    else:                base = 4.40

    mult = 1.0
    if self_employed:              mult *= 2.5
    if charitable_ratio > 0.20:    mult *= 1.8
    elif charitable_ratio > 0.10:  mult *= 1.35
    elif charitable_ratio > 0.05:  mult *= 1.15
    if has_business_loss:          mult *= 1.6
    if cash_business:              mult *= 1.4
    if large_deductions:           mult *= 1.25

    adjusted = base * mult
    score = min(100, round(np.log1p(adjusted * 10) / np.log1p(50) * 100, 1))
    level = "Low" if score < 25 else "Moderate" if score < 50 else "Elevated" if score < 70 else "High"
    return {"score": score, "level": level, "base_rate": f"{base:.2f}%",
            "adjusted_rate": f"{min(adjusted,15):.2f}%", "multiplier": f"{mult:.1f}x"}

profiles = [
    ("W-2, $65k",         dict(agi=65000)),
    ("W-2, $150k",        dict(agi=150000)),
    ("Freelancer, $90k",  dict(agi=90000, self_employed=True)),
    ("Small biz, $200k",  dict(agi=200000, self_employed=True, cash_business=True)),
    ("Biz+loss, $200k",   dict(agi=200000, self_employed=True, has_business_loss=True)),
    ("High earner, $750k",dict(agi=750000)),
    ("HE+Sch.C, $750k",   dict(agi=750000, self_employed=True, charitable_ratio=0.12)),
    ("Ultra-high, $5M",   dict(agi=5000000, charitable_ratio=0.15)),
]

rows = []
for name, params in profiles:
    r = audit_risk_score(**params)
    rows.append({"Profile": name, "Score": r["score"], "Level": r["level"],
                 "Base Rate": r["base_rate"], "Adjusted Rate": r["adjusted_rate"], "Multiplier": r["multiplier"]})
pd.DataFrame(rows)

## 6. Risk score visualization and sensitivity analysis

In [ ]:
df = pd.DataFrame(rows)
color_map = {"Low": GREEN, "Moderate": BLUE, "Elevated": AMBER, "High": RED}
bar_colors = [color_map[l] for l in df["Level"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("IRS audit risk — scores and sensitivity analysis", y=1.01)

bars = axes[0].barh(df["Profile"], df["Score"], color=bar_colors, edgecolor="#0e0f11")
for bar, s in zip(bars, df["Score"]):
    axes[0].text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 f"{s}", va="center", fontsize=9, color="#f0f0ee")
for thresh, color in [(25,GREEN),(50,BLUE),(70,AMBER)]:
    axes[0].axvline(thresh, color=color, linestyle="--", alpha=0.5)
axes[0].set_xlabel("Audit risk score (0-100)")
axes[0].set_xlim(0, 108)
axes[0].set_title("Risk scores by profile")
axes[0].grid(True, axis="x", alpha=0.3)

agi_range = np.linspace(20000, 2000000, 300)
for label, kwargs, color in [
    ("W-2 only", {}, BLUE),
    ("+Self-employed", dict(self_employed=True), AMBER),
    ("+SE+loss", dict(self_employed=True, has_business_loss=True), RED),
    ("+SE+15% charity", dict(self_employed=True, charitable_ratio=0.15), PURPLE),
]:
    scores = [audit_risk_score(a, **kwargs)["score"] for a in agi_range]
    axes[1].plot(agi_range/1000, scores, label=label, linewidth=2, color=color)
axes[1].set_xlabel("AGI ($k)")
axes[1].set_ylabel("Audit risk score (0-100)")
axes[1].set_title("Score sensitivity by filing profile")
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"${x:.0f}k"))
axes[1].legend(facecolor="#1c1e24", edgecolor="#3a3d45", labelcolor="#f0f0ee")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Key findings

- **Overall audit rates fell 56%** from 2010 (1.11%) to 2020 (0.29%) as IRS enforcement budgets were cut
- **High-income rates collapsed most**: filers with >$10M TPI went from 18.38% (2010) to 2.48% (2020)
- **Self-employment is the largest individual risk multiplier**: ~2.5x base audit probability
- **IRA recovery is real but partial**: 2022 overall rate (0.49%) remains less than half of 2010 levels
- **Compound risk is nonlinear**: a self-employed $750k filer with 12% charitable ratio faces 3.4x base rate

---

*Data: IRS Data Book Publication 55B Table 17 (Tax Years 2010–2022). Risk model weights derived from IRS DIF score documentation and TIGTA audit reports. For educational purposes only.*